# Task 2: End-to-End Customer Churn Pipeline

In [ ]:
!pip install pandas scikit-learn joblib matplotlib seaborn -q

## Load Data

In [ ]:
import pandas as pd

df=pd.read_csv('Telco-Customer-Churn.csv')
df.head()

## Preprocessing + Pipeline

In [ ]:
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder,StandardScaler
from sklearn.impute import SimpleImputer
from sklearn.model_selection import train_test_split,GridSearchCV
from sklearn.ensemble import RandomForestClassifier
from sklearn.linear_model import LogisticRegression

X=df.drop('Churn',axis=1)
y=df['Churn']

cat=X.select_dtypes(include='object').columns
num=X.select_dtypes(exclude='object').columns

pre=ColumnTransformer([
('num',Pipeline([('imp',SimpleImputer()),('scaler',StandardScaler())]),num),
('cat',Pipeline([('imp',SimpleImputer(strategy='most_frequent')),('enc',OneHotEncoder(handle_unknown='ignore'))]),cat)
])

## Logistic Regression

In [ ]:
pipe=Pipeline([('pre',pre),('model',LogisticRegression(max_iter=1000))])
params={'model__C':[0.1,1,10]}
search=GridSearchCV(pipe,params,cv=3,scoring='accuracy')
search.fit(X,y)
print(search.best_score_)

## Random Forest

In [ ]:
rf=Pipeline([('pre',pre),('model',RandomForestClassifier())])
rf_params={'model__n_estimators':[100,200]}
rf_search=GridSearchCV(rf,rf_params,cv=3)
rf_search.fit(X,y)

## Export Best Pipeline

In [ ]:
import joblib
best=search if search.best_score_>rf_search.best_score_ else rf_search
joblib.dump(best.best_estimator_,'churn_pipeline.pkl')

## Final Summary
Reusable production-ready ML pipeline with GridSearchCV and joblib export.